# Gemma Guide Demo Bootstrap

This notebook is the end-to-end bootstrap path for a fresh runtime.



## Quick Guide

Run this notebook top to bottom in a fresh runtime. It bootstraps the Gemma Guide demo, applies the required Kaggle/T4 compatibility workarounds, starts the demo stack, and gives you the public tunnel URL.

Recommended order:
1. Set `GITHUB_TOKEN` and `HF_TOKEN`.
2. Review the runtime and serving configuration.
3. Clone the repo and install dependencies.
4. Apply the required `vLLM` patches and clear caches.
5. Run the Hugging Face login cell if needed.
6. Start the demo stack, then inspect status and logs.
7. Run the cleanup cell when you are done.

Note:
- The notebook is tuned for Colab Tesla T4 runtimes and includes required compatibility fixes for that environment. These are workaround patches rather than upstream-approved changes, and they may reduce performance. Ideally, the demo should be run on a newer non-Turing NVIDIA GPU.


## Token Setup

Paste your tokens into the cell below before running the clone and Hugging Face login steps.

- `GITHUB_TOKEN` is needed if the repo is private.
- `HF_TOKEN` is needed if model download requires authenticated Hugging Face access.

In [1]:
import os

# Paste tokens here if they are not already set in the runtime environment.
GITHUB_TOKEN = ""
HF_TOKEN = ""

if GITHUB_TOKEN:
    os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

print('GITHUB_TOKEN set:', bool(os.getenv('GITHUB_TOKEN')))
print('HF_TOKEN set:', bool(os.getenv('HF_TOKEN')))

GITHUB_TOKEN set: True
HF_TOKEN set: True


## Runtime Config

In [2]:
from pathlib import Path
import os
import platform

REPO_URL = "https://github.com/pariidanDKE/GemmaGuide.git"
REPO_REF = "demo-setup"

if Path('/content').exists():
    REPO_DIR = Path('/content/GemmaGuide')
else:
    REPO_DIR = Path.cwd() / 'GemmaGuide'

HF_TOKEN = os.getenv('HF_TOKEN', '')
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')

if GITHUB_TOKEN and REPO_URL.startswith('https://github.com/'):
    CLONE_URL = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
else:
    CLONE_URL = REPO_URL

APP_PORT = os.getenv('APP_PORT', '7862')
VLLM_PORT = os.getenv('VLLM_PORT', '8000')

print(f'Python: {platform.python_version()}')
print(f'Platform: {platform.platform()}')
print(f'Repo dir: {REPO_DIR}')
print(f'Repo ref: {REPO_REF}')
print(f'App port: {APP_PORT}')
print(f'vLLM port: {VLLM_PORT}')

Python: 3.12.13
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Repo dir: /content/GemmaGuide
Repo ref: demo-setup
App port: 7862
vLLM port: 8000


## Serving Knobs

These are the main runtime knobs that affect whether the demo fits in memory and how much visual detail Gemma gets.

- `MODEL_CHOICE`: `e2b` is lighter and safer for tighter VRAM; `e4b` is stronger but heavier.
- `VLLM_QUANTIZATION`: use `fp8`, `bitsandbytes`, or `none`. `none` omits the quantization flag entirely.
- `MAX_SOFT_TOKENS`: visual token budget per image. Higher means more detail and more VRAM. Typical values: `140`, `280`, `560`.
- `MAX_MODEL_LEN`: context length. Lowering this is one of the cleanest VRAM reductions.
- `GPU_MEM_UTIL`: fraction of GPU memory vLLM is allowed to reserve. Too low can prevent startup.
- `MAX_NUM_SEQS`: keep this at `1` on smaller GPUs.
- `TIPS_SHORT_SIDE`: lower values reduce TIPS memory and latency.
- `VLLM_EXTRA_ARGS`: optional raw extra flags appended to the `vllm serve` command.


**Note** : For visual tasks, larger settings help. MAX_SOFT_TOKENS and TIPS_SHORT_SIDE both increase the visual detail available to the model, which tends to improve scene description and localization, at the cost of VRAM and speed. Likewise, gemma-4-E4B-it is generally more reliable than gemma-4-E2B-it for these tasks, but it is harder to fit on smaller GPUs.

In [4]:
MODEL_CHOICE = 'e4b'
MODEL_OPTIONS = {
    'e2b': ('google/gemma-4-E2B-it', 'gemma-4-e2b-it'),
    'e4b': ('google/gemma-4-E4B-it', 'gemma-4-e4b-it'),
}

VLLM_MODEL_REPO, VLLM_SERVED_NAME = MODEL_OPTIONS[MODEL_CHOICE]
VLLM_QUANTIZATION = 'bitsandbytes'
MAX_SOFT_TOKENS = '140'
MAX_MODEL_LEN = '4096'
GPU_MEM_UTIL = '0.75'
MAX_NUM_SEQS = '1'
TENSOR_PARALLEL = '1'
TIPS_SHORT_SIDE = '448'
VLLM_EXTRA_ARGS = '--enforce-eager'


print('model repo =', VLLM_MODEL_REPO)
print('served name =', VLLM_SERVED_NAME)
print('quantization =', VLLM_QUANTIZATION)
print('max_soft_tokens =', MAX_SOFT_TOKENS)
print('max_model_len =', MAX_MODEL_LEN)
print('gpu_mem_util =', GPU_MEM_UTIL)
print('max_num_seqs =', MAX_NUM_SEQS)
print('tensor_parallel =', TENSOR_PARALLEL)
print('tips_short_side =', TIPS_SHORT_SIDE)
print('vllm_extra_args =', VLLM_EXTRA_ARGS or '<none>')


model repo = google/gemma-4-E4B-it
served name = gemma-4-e4b-it
quantization = bitsandbytes
max_soft_tokens = 140
max_model_len = 4096
gpu_mem_util = 0.75
max_num_seqs = 1
tensor_parallel = 1
tips_short_side = 448
vllm_extra_args = --enforce-eager


## Clone The Repo

If the repo already exists in the runtime, this cell leaves it in place.

In [9]:
import subprocess

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', CLONE_URL, str(REPO_DIR)], check=True)

if REPO_REF:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)

subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], check=True)

CompletedProcess(args=['git', '-C', '/content/GemmaGuide', 'rev-parse', '--short', 'HEAD'], returncode=0)

## Install Dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q -r {REPO_DIR / 'requirements.txt'}

## Required vLLM Patches

Gemma Guide was developed on an RTX 3090 (Ampere, SM 8.6). The demo runs on a Tesla T4 (Turing, SM 7.5), which required two patches to vLLM's installed packages. These are workaround patches rather than upstream-approved changes, and they may reduce performance. Ideally, the demo should be run on a newer non-Turing NVIDIA GPU.

- **Triton tile size (`#38918`)**: Gemma 4's global attention layers have `head_dim=512`, requiring 96KB shared memory in vLLM's default Triton kernel. T4 is capped at 64KB. This patch reduces tile size from `32` to `16` for GPUs below SM 8.0, bringing usage within limits.
- **Audio encoder dtype**: vLLM falls back from `bfloat16` to `float16` on T4, but the audio encoder's `embedding_projection` layer was not getting cast. This patch explicitly casts audio encodings to match the projection weight dtype before the linear layer, mirroring what the vision encoder already did correctly.


In [ ]:
from pathlib import Path
import os
import shutil
import vllm


def apply_patch(filepath, old, new):
    with open(filepath, 'r') as f:
        content = f.read()

    assert old in content, f"Pattern not found in {filepath}!"
    content = content.replace(old, new, 1)

    with open(filepath, 'w') as f:
        f.write(content)


attention_patch_file = Path(vllm.__file__).resolve().parent / 'v1/attention/ops/triton_unified_attention.py'
attention_old = """    if _is_gemma3_attention(head_size, sliding_window):
        # Gemma3: use 32 for decode (default is 16)
        return 32"""
attention_new = """    if not current_platform.has_device_capability(80) and head_size > 128:
        return 16

    if _is_gemma3_attention(head_size, sliding_window):
        # Gemma3: use 32 for decode (default is 16)
        return 32"""
apply_patch(attention_patch_file, attention_old, attention_new)

audio_patch_file = Path(vllm.__file__).resolve().parent / 'model_executor/models/gemma4_mm.py'
audio_old = """        # Project into LM embedding space.
        audio_features = self.embed_audio(inputs_embeds=audio_encodings)"""
audio_new = """        # Project into LM embedding space.
        target_dtype = self.embed_audio.embedding_projection.weight.dtype
        audio_encodings = audio_encodings.to(target_dtype)
        audio_features = self.embed_audio(inputs_embeds=audio_encodings)"""
apply_patch(audio_patch_file, audio_old, audio_new)

for path in ['~/.triton', '~/.cache/vllm/torch_compile_cache']:
    expanded_path = os.path.expanduser(path)
    if os.path.exists(expanded_path):
        shutil.rmtree(expanded_path)
        print(f"Cleared {expanded_path}")

print("Patches applied successfully")


## Hugging Face Login

In [12]:
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged into Hugging Face using HF_TOKEN from the environment.')
else:
    print('HF_TOKEN is empty. If model download fails, set HF_TOKEN and rerun this cell.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged into Hugging Face using HF_TOKEN from the environment.


## Start The Demo Stack

This starts `vllm`, the app, and a Cloudflare Quick Tunnel in the background, then prints structured status as JSON.

The model is started as a separate server process on purpose. That keeps the app modular, preserves a clean OpenAI-compatible seam, and makes it easier to swap `vllm` for another backend later without rewriting the app flow.

Logs are written to `/tmp/gemma_demo_logs/`:
- `vllm.log`
- `app.log`
- `cloudflared.log`
- `demo_state.json`

In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
# Kaggle T4 runtimes expose CUDA 13.0 drivers but miss libnvJitLink.so.13, and bitsandbytes has no usable cuda130 binary for this setup.
# Forcing CUDA 12.9 makes bitsandbytes load its cuda129 binary, which works correctly against the 13.0 driver.
os.environ["BNB_CUDA_VERSION"] = "129"
env["BNB_CUDA_VERSION"] = "129"
env["VLLM_ATTENTION_BACKEND"] = "TORCH_SDPA"

start_cmd = [
    sys.executable,
    'scripts/demo_bootstrap.py',
    'start',
    '--with-tunnel',
    '--verbose',
    '--app-port', APP_PORT,
    '--vllm-port', VLLM_PORT,
    '--model', VLLM_MODEL_REPO,
    '--vllm-served-name', VLLM_SERVED_NAME,
    '--quantization', VLLM_QUANTIZATION,
    '--max-soft-tokens', MAX_SOFT_TOKENS,
    '--max-model-len', MAX_MODEL_LEN,
    '--gpu-mem-util', GPU_MEM_UTIL,
    '--max-num-seqs', MAX_NUM_SEQS,
    '--tensor-parallel', TENSOR_PARALLEL,
    '--tips-short-side', TIPS_SHORT_SIDE,
    '--dtype', 'float16',
]

if VLLM_EXTRA_ARGS:
    start_cmd.append(f'--vllm-extra-args={VLLM_EXTRA_ARGS}')

print(start_cmd)

bootstrap_proc = subprocess.Popen(
    start_cmd,
    cwd=str(REPO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)

print("bootstrap pid:", bootstrap_proc.pid)


['/usr/bin/python3', 'scripts/demo_bootstrap.py', 'start', '--with-tunnel', '--verbose', '--app-port', '7862', '--vllm-port', '8000', '--model', 'google/gemma-4-E4B-it', '--vllm-served-name', 'gemma-4-e4b-it', '--quantization', 'bitsandbytes', '--max-soft-tokens', '140', '--max-model-len', '4096', '--gpu-mem-util', '0.75', '--max-num-seqs', '1', '--tensor-parallel', '1', '--tips-short-side', '448', '--dtype', 'float16', '--vllm-extra-args=--enforce-eager']
bootstrap pid: 26293


In [ ]:
!tail -200 /tmp/gemma_demo_logs/vllm.log

## Inspect Status

In [15]:
import json
import subprocess
import sys

status_raw = subprocess.check_output(
    [sys.executable, 'scripts/demo_bootstrap.py', 'status'],
    cwd=REPO_DIR,
    text=True,
)
status = json.loads(status_raw)
print(json.dumps(status, indent=2))
print('\nOpen this URL on your phone:')
print(status.get('urls', {}).get('public'))

{
  "config": {
    "app_host": "127.0.0.1",
    "app_port": 7862,
    "dtype": "float16",
    "gpu_mem_util": "0.75",
    "max_model_len": "4096",
    "max_num_seqs": "1",
    "max_soft_tokens": "140",
    "model": "google/gemma-4-E4B-it",
    "quantization": "bitsandbytes",
    "tensor_parallel": "1",
    "tips_short_side": "448",
    "vllm_extra_args": "--enforce-eager",
    "vllm_port": 8000,
    "vllm_served_name": "gemma-4-e4b-it",
    "with_tunnel": true
  },
  "log_dir": "/tmp/gemma_demo_logs",
  "repo_root": "/content/GemmaGuide",
  "running": true,
  "services": {
    "vllm": {
      "log_path": "/tmp/gemma_demo_logs/vllm.log",
      "log_tail": "auncher.py:46] Route: /load, Methods: GET\n(APIServer pid=26294) INFO 05-17 21:14:50 [launcher.py:46] Route: /version, Methods: GET\n(APIServer pid=26294) INFO 05-17 21:14:50 [launcher.py:46] Route: /health, Methods: GET\n(APIServer pid=26294) INFO 05-17 21:14:50 [launcher.py:46] Route: /metrics, Methods: GET\n(APIServer pid=26294) I

In [12]:
import time
time.sleep(8)
!grep -i "trycloudflare.com" /tmp/gemma_demo_logs/cloudflared.log

2026-05-17T21:02:33Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-17T21:02:38Z INF |  https://tension-corps-merge-doctors.trycloudflare.com                                     |
2026-05-17T21:02:57Z ERR Request failed error="stream 45 canceled by remote with error code 0" connIndex=0 dest=https://tension-corps-merge-doctors.trycloudflare.com/designs/audio/misc_audio/model_load.mp3 event=0 ip=198.41.192.67 type=http


## Log Helpers

In [ ]:
from pathlib import Path
import json

state_path = Path('/tmp/gemma_demo_logs/demo_state.json')
state = json.loads(state_path.read_text()) if state_path.exists() else {}

def tail_file(path, lines=80):
    path = Path(path)
    if not path.exists():
        print(f'{path} does not exist')
        return
    content = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    print('\n'.join(content[-lines:]))

for name, info in state.get('services', {}).items():
    print(f'=== {name} :: {info.get("log_path")} ===')
    tail_file(info['log_path'], lines=40)
    print()

## Cleanup

In [69]:
import subprocess
import sys

subprocess.run([sys.executable, 'scripts/demo_bootstrap.py', 'stop'], cwd=REPO_DIR, check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/demo_bootstrap.py', 'stop'], returncode=0)